# Challenge videos: detections and jersey numbers

**Setup:** Runtime > Change runtime type > **A100 GPU**, then Runtime > **Run all**. Keep this tab open.

1. Allow the Google Drive prompt. The videos come from `My Drive/test_videos` (any subfolder): `IMG_0114.MP4`
   (150,358,487 bytes) and `IMG_0115.MP4` (154,342,609 bytes). A file of
   another size is skipped as an incomplete upload.
2. The weights are found in Drive by their hash, as in `jnr_all_videos.ipynb`:
   - the detector from `tracking_results/src_train_detector/`;
   - the eligibility head and ViTSTR from `tracking_results/src_train_classifier/`, or any copy under
     `My Drive/tracking_weights`.
   They are pinned by sha256 to the checkpoints the training-game results were made with (readability head
   `4d403c88`). If one is missing, the notebook stops rather than use other weights.
3. Per video:
   - the fine-tuned RF-DETR Small runs on every frame, with 8 preview images;
   - every detection with confidence ≥ 0.3 on every 2nd frame gets a ViTPose torso crop, a
     readability score and a score for each of the 9 numbers.
4. Results go to `My Drive/tracking_results/<video>/<code version>/`, under `detections/` and `jnr/`, one video at a
   time. Run all again skips finished videos. At the end `jnr_test_<code version>.zip` is downloaded and kept in
   `My Drive/tracking_results/`.

**Rules:** nothing here trains. All models were fine-tuned on the training games only, and the challenge videos are
only read for inference. Bring the zip back to the Mac for colour labels, tracking, stitching and the submission.

In [ ]:
import os, shutil, subprocess, sys, time, hashlib, zipfile
from pathlib import Path

VIDEOS = {'IMG_0114.MP4': 150358487, 'IMG_0115.MP4': 154342609}
WEIGHTS_SHA256 = "471b3fff3f77d320f9c84cdc1d8978d8a1b6162d6813915c1d8e0c700955f80a"
PREVIEWS = 8
MODEL = "rfdetr-small-ft-" + WEIGHTS_SHA256[:8]
CODE_VERSION = "0ddd8626"
REPO = Path("/content/tracking_challenge")
PY = REPO / ".venv/bin/python"
OUTPUTS = REPO / "outputs"

def sh(cmd, cwd=REPO, log=None):
    """Run a command, stream its output, optionally keep a copy, and raise if it fails."""
    print("$", " ".join(map(str, cmd)), flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "MPLBACKEND": "agg"}  # Colab's inline backend does not exist in our env
    p = subprocess.Popen(list(map(str, cmd)), cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = open(log, "a") if log else None
    for line in p.stdout:
        print(line, end="", flush=True)
        if out:
            out.write(line)
    if out:
        out.close()
    if p.wait():
        raise RuntimeError(f"command failed with exit code {p.returncode}: {cmd}")

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert gpu.returncode == 0, "No GPU. Set Runtime > Change runtime type > A100 GPU, then run again."
print(gpu.stdout.strip())
if "A100" not in gpu.stdout:
    print("WARNING: this is not an A100. The run still works but will be slower.")

In [ ]:
# Find the three weights and every video before spending any GPU time.
from google.colab import drive, files

drive.mount("/content/drive")
MY = Path("/content/drive/MyDrive")
DRIVE_DIR = MY / "test_videos"
RESULTS_ROOT = MY / "tracking_results"
assert DRIVE_DIR.is_dir(), f"{DRIVE_DIR} not found. Check the folder name and that it is in this account's My Drive."

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

WEIGHTS_USED = {}

def fetch_weights(name, digest, folders, pattern):
    """The file with exactly this sha256, or stop: results must come from the pinned weights and no others."""
    local = Path("/content/weights") / name
    if local.exists() and sha256(local) == digest:
        WEIGHTS_USED[name] = {"sha256": digest, "expected": True}
        return local
    places = [p for d in folders if d.is_dir() for p in sorted(d.rglob(pattern))]
    src = next((p for p in places if p.is_file() and p.stat().st_size > 1e6 and sha256(p) == digest), None)
    assert src, f"No {name} with sha256 {digest[:8]}... in {[str(d) for d in folders]}. Put a copy in My Drive/tracking_weights."
    local.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(src, local)
    got = sha256(local)
    assert got == digest, f"{name}: copy of {src} has sha256 {got[:8]}, expected {digest[:8]}"
    WEIGHTS_USED[name] = {"source": str(src), "sha256": got, "expected": True}
    print(f"{name}: {src} (sha256 {got[:8]})")
    return local

WEIGHTS = fetch_weights("detector.pth", WEIGHTS_SHA256, [MY / "tracking_weights", RESULTS_ROOT / "src_train_detector"], "*.pth")
ELIG = fetch_weights("eligibility.pt", "4d403c88ad87819e86a4e7b3a79dfdaeae09369d8342ec93725df6dd74dd43f2", [MY / "tracking_weights", RESULTS_ROOT / "src_train_classifier"], "*.pt")
VITSTR = fetch_weights("vitstr.pt", "7319ecec70eb2d5b188c4cc161611ee9faf65d666d6182a39e3378a7387734d6", [MY / "tracking_weights", RESULTS_ROOT / "src_train_classifier"], "*.pt")

on_drive = [p for p in DRIVE_DIR.rglob("*") if p.is_file()]
SOURCES, problems = {}, {}
for name, size in VIDEOS.items():
    found = [p for p in on_drive if p.name.lower() == name.lower()]
    complete = [p for p in found if p.stat().st_size == size]
    if complete:
        SOURCES[name] = complete[0]
    elif found:
        problems[name] = f"expected {size} bytes, found {[p.stat().st_size for p in found]}. The upload may be incomplete."
    else:
        problems[name] = "not in the folder."
print(f"Ready: {len(SOURCES)} of {len(VIDEOS)} videos")
for name, why in problems.items():
    print(f"  SKIPPING {name}: {why}")
assert SOURCES, f"No usable videos under {DRIVE_DIR}. Files there: {sorted(p.name for p in on_drive)}"

In [ ]:
# Code bundled by colab/build_jnr_notebook.py
import base64, io, tarfile
BUNDLE = "..."
REPO.mkdir(exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE))) as tar:
    tar.extractall(REPO, filter="data")
print("Unpacked; code version", CODE_VERSION)

In [ ]:
# Same locked versions as the local repo, with the rfdetr group, in a separate env so Colab's packages don't interfere.
EXCLUDE = ['python-doctr', 'rf-groundingdino']
sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
sh([sys.executable, "-m", "uv", "sync", "--frozen", "--no-dev", "--python", "3.12", "--group", "rfdetr",
    *[arg for package in EXCLUDE for arg in ("--no-install-package", package)]])
sh([PY, "-c", "import torch, rfdetr, cv2; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); "
    "assert torch.cuda.is_available(), 'torch cannot see the GPU'"])

In [ ]:
# One video at a time: detections (reused or made), then jersey numbers; saved to Drive as soon as each finishes.
import json
EVERY, MIN_CONF = 2, 0.3

def save(src, dest):
    """Copy to Drive. The rename means a partial copy never looks complete."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    partial = dest.with_name(dest.name + ".partial")
    shutil.copyfile(src, partial)
    os.replace(partial, dest)

def existing_dets(stem):
    """Fine-tuned detections for this video from any earlier run in Drive, newest first."""
    d = RESULTS_ROOT / stem
    hits = sorted(d.glob(f"*/detections/{stem}__{MODEL}__720p.txt"), key=lambda p: p.stat().st_mtime, reverse=True) if d.is_dir() else []
    return hits[0] if hits else None

def same_weights(results):
    """True if this folder's results were made with the weights loaded now (from its weights_used.json)."""
    try:
        old = json.loads((results / "reports/weights_used.json").read_text())
    except (OSError, ValueError):
        return False
    return all(old.get(k, {}).get("sha256") == v.get("sha256") for k, v in WEIGHTS_USED.items())

def package():
    """One zip of every jnr file, new detection file, preview and log from this code version; to Drive and downloaded."""
    z_path = Path(f"/content/jnr_test_{CODE_VERSION}.zip")
    with zipfile.ZipFile(z_path, "w", zipfile.ZIP_DEFLATED, strict_timestamps=False) as z:
        for name in VIDEOS:
            d = RESULTS_ROOT / Path(name).stem / CODE_VERSION
            for f in sorted(d.rglob("*")) if d.is_dir() else []:
                if f.is_file() and f.suffix != ".partial":
                    z.write(f, f"{Path(name).stem}/{f.relative_to(d)}")
    print(f"built {z_path.name} ({z_path.stat().st_size / 1e6:.1f} MB)")
    try:
        save(z_path, RESULTS_ROOT / z_path.name)
        drive.flush_and_unmount()  # Drive uploads in the background; make sure everything has reached it
        print("Results are in Drive:", RESULTS_ROOT)
    except Exception as e:
        print("Copy to Drive failed:", repr(e), "- the zip is at", z_path)
    try:
        files.download(str(z_path))
    except Exception as e:
        print(f"Download failed ({e!r}). Get it from Drive: {RESULTS_ROOT / z_path.name}")

failures = {}
try:
    for name, source in SOURCES.items():
        stem = Path(name).stem
        results = RESULTS_ROOT / stem / CODE_VERSION
        dets_name = f"{stem}__{MODEL}__720p.txt"
        jnr_name = f"jnr/{stem}__{MODEL}__720p__jnr.csv.gz"
        if (results / "done/jnr").exists() and (results / jnr_name).exists():
            if same_weights(results):
                print(f"=== {stem}: already done, skipping", flush=True)
                continue
            print(f"=== {stem}: earlier results used other weights, redoing", flush=True)
            (results / "done/jnr").unlink()
        print(f"=== {stem}", flush=True)
        start = time.time()
        local = Path("/content") / name
        log = OUTPUTS / "reports" / stem / "jnr_log.txt"
        log.parent.mkdir(parents=True, exist_ok=True)
        dets = OUTPUTS / "detections" / dets_name
        try:
            shutil.copyfile(source, local)
            assert local.stat().st_size == VIDEOS[name], f"{local} has the wrong size after copying"
            found = existing_dets(stem)
            if found is not None:
                print(f"{stem}: detections from {found}", flush=True)
                dets.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(found, dets)
                # keep the exact detections the jersey reads point into, and where they came from, with this run
                save(dets, results / "detections" / dets_name)
                prov = OUTPUTS / "reports" / stem / "detections_source.json"
                prov.write_text(json.dumps({"reused_from": str(found), "sha256": sha256(dets)}, indent=1))
                save(prov, results / "reports/detections_source.json")
            else:
                print(f"{stem}: no fine-tuned detections in Drive yet, running the detector", flush=True)
                sh([PY, "-m", "src.infer.detect_video", "--video", local, "--weights", WEIGHTS, "--previews", 8], log=log)
                save(dets, results / "detections" / dets_name)
                for p in sorted((OUTPUTS / "previews").glob(f"{stem}__*.jpg")):
                    save(p, results / "previews" / p.name)
            sh([PY, "-m", "src.infer.jnr_video", "--video", local, "--dets", dets, "--eligibility", ELIG, "--vitstr", VITSTR,
                "--every", EVERY, "--min-conf", MIN_CONF], log=log)
            save(OUTPUTS / "jnr" / Path(jnr_name).name, results / jnr_name)
            used = OUTPUTS / "reports" / stem / "weights_used.json"
            used.write_text(json.dumps(WEIGHTS_USED, indent=1))
            save(used, results / "reports/weights_used.json")
            save(log, results / "reports/jnr_log.txt")
            (results / "done").mkdir(parents=True, exist_ok=True)
            (results / "done/jnr").touch()
        except Exception as e:
            failures[name] = str(e)
            print(f"{stem} FAILED: {e}", flush=True)
            if log.exists():
                save(log, results / "reports/jnr_log.txt")
        finally:
            local.unlink(missing_ok=True)
            print(f"{stem}: {(time.time() - start) / 60:.1f} min", flush=True)
finally:
    package()

unfinished = {**problems, **failures}
if unfinished:
    raise RuntimeError("Not finished, everything else is in Drive. Fix these, then choose Run all to process only what is missing:\n"
                       + "\n".join(f"  {name}: {why}" for name, why in unfinished.items()))
print(f"All {len(VIDEOS)} videos are done.")